# Week 7 — LangGraph Challenge

**Module 3 Unit 2** | Build, guard, and observe a minimal production-minded LangGraph agent.

`src/chains/memory_agent.py` (the completed Week 3 agent) uses `langgraph.prebuilt.create_react_agent`,
which hides the agent/tool loop behind a single call. Its module docstring frames that as a stepping
stone: *"create_react_agent from langgraph.prebuilt is the current approach ... making it easy to
inspect, extend, and eventually swap for a custom StateGraph in Module 3."* This notebook is that swap.

Instead of a prebuilt loop, this notebook hand-builds the graph so every piece of production behavior
is visible and testable: typed state with reducers, an explicit loop-cap guard, input validation that
never raises, checkpointing with a `thread_id`, and streamed observability of every node transition.

| Component | What | Why |
|-----------|------|-----|
| State Schema | `TypedDict` with `add_messages` / `operator.add` reducers | Keeps state minimal (ids/counters, not payloads) while making concurrent updates well-defined |
| Tools | `add`, `multiply` — loosely-typed, never raise | Matches this repo's `customer_tools.py` convention: tools return friendly strings, not exceptions |
| Graph | `agent` node + `ToolNode` + `safe_exit` node | One LLM node, one tool node, wired with `tools_condition` plus a loop-cap guard |
| Guard | `route_after_agent` conditional edge, `MAX_TOOL_CALLS = 5` | Only conditional edges choose the next node in LangGraph — the cap lives there, not in a node |
| Checkpointing | `InMemorySaver` + `thread_id` | Demonstrates resuming a conversation and replaying a paused run |
| Observability | `app.stream(..., stream_mode="updates")` | Per-node event log showing exactly which edges fired and why |

All graph-building code lives in [`src/chains/langgraph_challenge_agent.py`](src/chains/langgraph_challenge_agent.py)
so it's reusable and independently unit-tested in
[`tests/test_langgraph_challenge_agent.py`](tests/test_langgraph_challenge_agent.py) (offline, with a fake
LLM double — no API calls). This notebook imports that module and drives it against the real model for
the graded run logs below.


## Cell 1 — Setup

In [1]:
import os

from dotenv import load_dotenv
load_dotenv()

# LangSmith tracing env vars must be set before any ChatOpenAI is instantiated
# (LangChain reads them at creation time) — same pattern as
# TechStorePlus_LangChain_LCEL_Chatbot.ipynb.
os.environ.setdefault("LANGCHAIN_PROJECT", "Week7-LangGraph-Challenge")
if os.getenv("LANGCHAIN_API_KEY", "").strip():
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    print("LangSmith tracing ENABLED — project:", os.environ["LANGCHAIN_PROJECT"])
else:
    print("LangSmith tracing disabled (no LANGCHAIN_API_KEY) — the graph still runs normally.")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env"

from langchain_core.messages import HumanMessage

from src.chains.langgraph_challenge_agent import (
    MAX_TOOL_CALLS,
    TOOLS,
    add,
    multiply,
    build_graph,
    initial_state,
)

print("Tools bound to the agent:", [t.name for t in TOOLS])
print("Loop cap: MAX_TOOL_CALLS =", MAX_TOOL_CALLS)


LangSmith tracing ENABLED — project: Advanced-Customer-Agent


Tools bound to the agent: ['add', 'multiply']
Loop cap: MAX_TOOL_CALLS = 5


---
## Component 1 — State Schema

**Why a reducer instead of manual list append?** Without a reducer, LangGraph's default behavior for a
`TypedDict` field is to *overwrite* it with whatever a node returns. `agent` and `tools` both run many
times per conversation turn, each returning only the small delta it produced (one new message, one
incremented counter) — not the full accumulated state. Reducers tell LangGraph how to merge each delta
into what's already there.


In [2]:
import inspect
from src.chains import langgraph_challenge_agent as w7

print(inspect.getsource(w7.AgentState))


class AgentState(TypedDict):
    """Minimal state: message history plus small counters/ids, not large payloads."""

    messages: Annotated[list[BaseMessage], add_messages]
    tool_calls: Annotated[int, operator.add]
    retries: Annotated[int, operator.add]
    errors: Annotated[list[str], operator.add]



- `messages: Annotated[list[BaseMessage], add_messages]` — `add_messages` appends new messages
  and de-duplicates by message id, which is what a conversational message log needs.
- `tool_calls: Annotated[int, operator.add]` — a running total of tool calls the agent has *requested*
  this thread, summed across every `agent` node execution. This is the value the loop-cap guard checks.
- `retries: Annotated[int, operator.add]` — a running total of transient-error retries (see Component 2
  and the write-up for why this stays `0` in this notebook's tools).
- `errors: Annotated[list[str], operator.add]` — `operator.add` on two lists is list concatenation, so
  each node's newly-observed errors are appended to the thread's running error log.
- State is kept minimal per the spec: no large payloads, just messages and small counters/strings.


---
## Component 2 — Tools

**Why loosely-typed parameters (`str | float | int`) instead of strict `float`?** LangChain
auto-generates a Pydantic schema from a `@tool` function's type hints. If `a`/`b` were typed as `float`,
a bad value like `"abc"` would be rejected by that generated schema *before* the tool body ever runs —
and the resulting error message would be framework-authored, not ours. This repo's `customer_tools.py`
establishes the convention that tools never raise; they always return a human-readable string so the
agent can continue the conversation. Loosening the parameter types lets the tool body validate and
produce that friendly string itself.

Error strings use a machine-parseable `"ERROR:<CLASS>: <detail>"` sentinel (`CLASS` is `VALIDATION` or
`TRANSIENT`) so the `agent` node can turn a friendly-but-failed tool result into a structured `errors`
state entry — see the write-up for how this reconciles "tools never raise" with the assignment's
requirement for a structured error list.


In [3]:
print(inspect.getsource(w7.add.func))
print(inspect.getsource(w7.multiply.func))


@tool
def add(a: str | float | int, b: str | float | int) -> str:
    """Add two numbers and return their sum.

    Args:
        a: The first addend (a number, e.g. 2.5, or a numeric string, e.g. "2.5").
        b: The second addend (a number, e.g. 7, or a numeric string, e.g. "7").
    """
    parsed_a, error_a = _parse_number(a)
    if error_a:
        return f"ERROR:VALIDATION: invalid value for a - {error_a}"
    parsed_b, error_b = _parse_number(b)
    if error_b:
        return f"ERROR:VALIDATION: invalid value for b - {error_b}"
    return str(parsed_a + parsed_b)

@tool
def multiply(a: str | float | int, b: str | float | int) -> str:
    """Multiply two numbers and return their product.

    Args:
        a: The first factor (a number, e.g. 2.5, or a numeric string, e.g. "2.5").
        b: The second factor (a number, e.g. 7, or a numeric string, e.g. "7").
    """
    parsed_a, error_a = _parse_number(a)
    if error_a:
        return f"ERROR:VALIDATION: invalid value for a -

In [4]:
print("Valid call:  ", add.invoke({"a": 2.5, "b": 7}))
print("Valid call:  ", multiply.invoke({"a": 9.5, "b": 3}))
print("Invalid call:", multiply.invoke({"a": "abc", "b": 5}))


Valid call:   9.5
Valid call:   28.5
Invalid call: ERROR:VALIDATION: invalid value for a - could not parse 'abc' as a number


---
## Component 3 — Graph Construction

**Where does the loop-cap check live — precisely?** In the conditional-edge function
`route_after_agent`, not inside a node. Node functions in LangGraph return state deltas; only a
conditional edge chooses the next node. The `tool_calls` counter is incremented inside `agent_node` at
the moment the LLM *proposes* new tool calls (before they execute). Because LangGraph applies a node's
state delta before evaluating its outgoing conditional edge, `route_after_agent` always sees the
up-to-date cumulative total:

```python
def route_after_agent(state: AgentState) -> str:
    base_route = tools_condition(state)   # "tools" or "__end__"
    if base_route != "tools":
        return END
    if state["tool_calls"] >= MAX_TOOL_CALLS:
        return "safe_exit"
    return "tools"
```

This means an over-limit tool call is abandoned *before* it ever runs — safe, because `safe_exit → END`
means no further agent turn ever needs its result.

Edges: `START → agent`; `agent → (tools | safe_exit | END)` via `route_after_agent`; `tools → agent`
(to feed results back); `safe_exit → END` (the safe-exit path required when the guard trips).


In [5]:
app = build_graph()  # real ChatOpenAI(gpt-4o-mini) + a fresh InMemorySaver by default

graph_view = app.get_graph()
print("Nodes:", list(graph_view.nodes))
print("\nEdges:")
for edge in graph_view.edges:
    tag = " (conditional)" if edge.conditional else ""
    print(f"  {edge.source:10s} -> {edge.target:10s}{tag}")


Nodes: ['__start__', 'agent', 'tools', 'safe_exit', '__end__']

Edges:
  __start__  -> agent     
  agent      -> __end__    (conditional)
  agent      -> safe_exit  (conditional)
  agent      -> tools      (conditional)
  tools      -> agent     
  safe_exit  -> __end__   


---
## Component 4 — Checkpointing & Thread IDs

Every run below is invoked with `config={"configurable": {"thread_id": ...}}`. Because state reducers
are additive and scoped to a checkpoint, **each demo/test below uses its own distinct `thread_id`** —
reusing one across cells would silently carry over stale `tool_calls`/`errors` counts from a prior run
and corrupt the `MAX_TOOL_CALLS` comparison.

This cell demonstrates the simplest form of checkpointing: two separate `.invoke()` calls on the same
`thread_id`, where the second turn has full context of the first without the caller re-sending history.


In [6]:
demo_config = {"configurable": {"thread_id": "week7-demo-checkpointing"}}

turn_1 = app.invoke(initial_state("Add 10 and 5."), config=demo_config)
print("Turn 1 final message:", turn_1["messages"][-1].content)
print("Turn 1 message count:", len(turn_1["messages"]))

# Same thread_id: LangGraph reloads the checkpointed history from turn 1 and appends
# this new human message, so the agent has full context of turn 1 without us resending it.
turn_2 = app.invoke(initial_state("Now multiply that result by 2."), config=demo_config)
print("\nTurn 2 final message:", turn_2["messages"][-1].content)
print("Turn 2 message count (cumulative):", len(turn_2["messages"]))
assert len(turn_2["messages"]) > len(turn_1["messages"]), "turn 2 should build on turn 1's checkpointed history"


Turn 1 final message: The sum of 10 and 5 is 15.
Turn 1 message count: 4



Turn 2 final message: The result of multiplying 15 by 2 is 30.
Turn 2 message count (cumulative): 8


---
## Component 5 — Acceptance Tests

### Acceptance Test 1 — Math chain

Prompt: *"Add 2.5 and 7, then multiply by 3."* Expected to use both tools and return an explanation.

> **Note on the expected value:** the challenge spec's own worked example states this should return
> `27.0`. That's arithmetically inconsistent — `(2.5 + 7) * 3 = 28.5`, not `27.0` (verified independently:
> `2.5 + 7 = 9.5`, and `9.5 * 3 = 28.5`). This notebook validates the mathematically correct result,
> `28.5`, and the tool composition (both `add` and `multiply` invoked in the right order) rather than the
> spec's literal number. The offline test suite documents the same discrepancy
> (`tests/test_langgraph_challenge_agent.py::test_math_chain_uses_both_tools_and_returns_28_5`).


In [7]:
test1_config = {"configurable": {"thread_id": "week7-accept-1-math-chain"}}

test1_states = list(app.stream(
    initial_state("Add 2.5 and 7, then multiply by 3."),
    config=test1_config,
    stream_mode="values",
))
final_state_1 = test1_states[-1]

print("Final answer:", final_state_1["messages"][-1].content)
print("Tool calls used:", final_state_1["tool_calls"])
print("Errors:", final_state_1["errors"])
assert "28.5" in final_state_1["messages"][-1].content


Final answer: The sum of 2.5 and 7 is 9.5. When you multiply that by 3, the result is 28.5.
Tool calls used: 3
Errors: []


### Acceptance Test 2 — Loop cap

A prompt engineered to provoke repeated tool calls, forcing the graph to exit via the `safe_exit` guard
rather than looping indefinitely.


In [8]:
test2_config = {"configurable": {"thread_id": "week7-accept-2-loop-cap"}}

test2_prompt = (
    "I want you to call the add tool over and over: start with 1, add 1 to get the next number, "
    "then take that result and add 1 again, and keep repeating this — call the add tool again and "
    "again for at least 10 rounds total. Do not give a final answer or stop early; keep refining "
    "the number and calling the tool again."
)

test2_states = list(app.stream(initial_state(test2_prompt), config=test2_config, stream_mode="values"))
final_state_2 = test2_states[-1]

print("Final message:", final_state_2["messages"][-1].content)
print("Tool calls used:", final_state_2["tool_calls"], "(cap =", MAX_TOOL_CALLS, ")")
print("Errors:", final_state_2["errors"])
assert final_state_2["tool_calls"] <= MAX_TOOL_CALLS
assert any(e.startswith("guard=loop_cap") for e in final_state_2["errors"]), (
    "expected the loop-cap guard to trip for this prompt"
)


Final message: I've hit the tool-call safety limit (5) for this turn. To keep things reliable, I'm stopping here rather than looping indefinitely. Please rephrase your request more specifically, or ask me to continue.
Tool calls used: 5 (cap = 5 )
Errors: ['guard=loop_cap tool_calls=5 limit=5']


### Acceptance Test 3 — Invalid input

`multiply(a="abc", b=5)` should be rejected with a clear validation error and a safe exit — without the
process ever raising an exception. Two views: the tool called directly, and the full graph handling the
same bad input inside a real conversation turn.


In [9]:
direct_result = multiply.invoke({"a": "abc", "b": 5})
print("Direct tool call result:", direct_result)
assert direct_result.startswith("ERROR:VALIDATION")


Direct tool call result: ERROR:VALIDATION: invalid value for a - could not parse 'abc' as a number


In [10]:
test3_config = {"configurable": {"thread_id": "week7-accept-3-invalid-input"}}

test3_prompt = (
    "Call the multiply tool with a set literally to the text 'abc' and b set to 5, even though "
    "'abc' is not a number. I want to see what happens."
)

test3_states = list(app.stream(initial_state(test3_prompt), config=test3_config, stream_mode="values"))
final_state_3 = test3_states[-1]

print("Final message:", final_state_3["messages"][-1].content)
print("Errors:", final_state_3["errors"])
# This is the graceful END path (agent explains the tool error in plain language) — distinct from
# the safe_exit NODE that only the loop-cap guard (Test 2) routes through.
assert any(e.startswith("tool=multiply class=VALIDATION") for e in final_state_3["errors"])


Final message: The attempt to use the multiply tool with 'abc' as the first argument resulted in an error because 'abc' cannot be parsed as a number. The tool requires both inputs to be valid numbers.
Errors: ["tool=multiply class=VALIDATION detail=invalid value for a - could not parse 'abc' as a number"]


### Acceptance Test 4 — Replay

Resume a partially completed run via the same `thread_id`, using `interrupt_before=["tools"]` to pause
right before the pending tool call executes, then show continuity after resuming.


In [11]:
test4_config = {"configurable": {"thread_id": "week7-accept-4-replay"}}
test4_app = build_graph(interrupt_before=["tools"])

test4_app.invoke(initial_state("Add 3 and 4."), config=test4_config)
paused_state = test4_app.get_state(test4_config)

print("Paused before node(s):", paused_state.next)
pending_tool_calls = paused_state.values["messages"][-1].tool_calls
print("Pending tool call:", pending_tool_calls)
assert paused_state.next == ("tools",)

# Resume the SAME thread_id with no new input — LangGraph continues from the checkpoint.
test4_final = test4_app.invoke(None, config=test4_config)
print("\nFinal message after resume:", test4_final["messages"][-1].content)

resumed_state = test4_app.get_state(test4_config)
print("Graph finished (no pending nodes):", resumed_state.next == ())
assert resumed_state.next == ()

tool_message = next(
    m for m in test4_final["messages"]
    if getattr(m, "tool_call_id", None) == pending_tool_calls[0]["id"]
)
print("Tool result for the resumed call:", tool_message.content)


Paused before node(s): ('tools',)
Pending tool call: [{'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'call_110qJEojDJOW1FLSHU8zruM6', 'type': 'tool_call'}]



Final message after resume: The sum of 3 and 4 is 7.
Graph finished (no pending nodes): True
Tool result for the resumed call: 7.0


---
## Component 6 — Observability (Streaming + Logs)

`app.stream(..., stream_mode="updates")` yields one event per node execution: `{node_name: state_delta}`.
Below are two full event-stream traces — one successful run (Acceptance Test 1's prompt) and one
guarded-exit run (Acceptance Test 2's prompt) — each on a fresh `thread_id` so the counts start at zero,
followed by a narrative of which edges fired and why.


In [12]:
def print_stream_log(prompt: str, thread_id: str) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    final_state = None
    for step_index, update in enumerate(app.stream(initial_state(prompt), config=config, stream_mode="updates")):
        for node_name, delta in update.items():
            summary_bits = []
            if "tool_calls" in delta:
                summary_bits.append(f"tool_calls+={delta['tool_calls']}")
            if "errors" in delta and delta["errors"]:
                summary_bits.append(f"errors+={delta['errors']}")
            print(f"[step {step_index}] node={node_name!r} {' '.join(summary_bits)}")
            for msg in delta.get("messages", []):
                kind = type(msg).__name__
                text = getattr(msg, "content", "") or ""
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    print(f"           {kind} requests: {[(c['name'], c['args']) for c in tool_calls]}")
                else:
                    print(f"           {kind}: {text[:100]}")
        final_state = app.get_state(config).values
    return final_state


**Run 1 — normal (successful) run:**

In [13]:
normal_run_final = print_stream_log(
    "Add 2.5 and 7, then multiply by 3.",
    thread_id="week7-observability-normal-run",
)
print("\n=== Final state ===")
print("Answer:", normal_run_final["messages"][-1].content)
print("tool_calls:", normal_run_final["tool_calls"], "| errors:", normal_run_final["errors"])


[step 0] node='agent' tool_calls+=2
           AIMessage requests: [('add', {'a': 2.5, 'b': 7}), ('multiply', {'a': 3, 'b': 3})]
[step 1] node='tools' 
           ToolMessage: 9.5
           ToolMessage: 9.0


[step 2] node='agent' tool_calls+=1
           AIMessage requests: [('multiply', {'a': 9.5, 'b': 3})]
[step 3] node='tools' 
           ToolMessage: 28.5


[step 4] node='agent' tool_calls+=0
           AIMessage: The sum of 2.5 and 7 is 9.5. When you multiply that by 3, the result is 28.5.

=== Final state ===
Answer: The sum of 2.5 and 7 is 9.5. When you multiply that by 3, the result is 28.5.
tool_calls: 3 | errors: []


**Narrative — normal run:** `START → agent` fires once; the LLM issues **two parallel tool calls** in a
single response — `add(2.5, 7)` and a speculative `multiply(3, 3)` guessed before the sum was known.
`route_after_agent` sees `tools_condition == "tools"` and `tool_calls (2) < MAX_TOOL_CALLS (5)`, so it
routes to `tools`; both calls execute in the same step, returning `9.5` and `9.0`. `tools → agent` feeds
both results back; the LLM recognizes its guess used the wrong operands and issues a corrected
`multiply(9.5, 3)` — the guard checks again (`tool_calls (3) < 5`) and routes to `tools` once more,
returning `28.5`. `tools → agent` feeds that back; this time the LLM has enough information to answer in
plain text (no `tool_calls` on the response), so `tools_condition` itself returns `END` before the cap is
even checked. Three tool calls total (one of them a self-corrected speculative guess), well under the
cap — the guard never had to intervene, and the final answer is still correct because the agent used the
real, tool-computed sum for its final `multiply` call rather than trusting its own guess.


**Run 2 — guarded-exit run:**

In [14]:
guarded_run_final = print_stream_log(
    test2_prompt,
    thread_id="week7-observability-guarded-exit-run",
)
print("\n=== Final state ===")
print("Final message:", guarded_run_final["messages"][-1].content)
print("tool_calls:", guarded_run_final["tool_calls"], "| errors:", guarded_run_final["errors"])


[step 0] node='agent' tool_calls+=1
           AIMessage requests: [('add', {'a': 1, 'b': 1})]
[step 1] node='tools' 
           ToolMessage: 2.0


[step 2] node='agent' tool_calls+=1
           AIMessage requests: [('add', {'a': 2, 'b': 1})]
[step 3] node='tools' 
           ToolMessage: 3.0


[step 4] node='agent' tool_calls+=1
           AIMessage requests: [('add', {'a': 3, 'b': 1})]
[step 5] node='tools' 
           ToolMessage: 4.0


[step 6] node='agent' tool_calls+=1
           AIMessage requests: [('add', {'a': 4, 'b': 1})]
[step 7] node='tools' 
           ToolMessage: 5.0


[step 8] node='agent' tool_calls+=1
           AIMessage requests: [('add', {'a': 5, 'b': 1})]
[step 9] node='safe_exit' errors+=['guard=loop_cap tool_calls=5 limit=5']
           AIMessage: I've hit the tool-call safety limit (5) for this turn. To keep things reliable, I'm stopping here ra

=== Final state ===
Final message: I've hit the tool-call safety limit (5) for this turn. To keep things reliable, I'm stopping here rather than looping indefinitely. Please rephrase your request more specifically, or ask me to continue.
tool_calls: 5 | errors: ['guard=loop_cap tool_calls=5 limit=5']


**Narrative — guarded-exit run:** `agent → tools` fires repeatedly as the LLM keeps proposing another
`add` call per the prompt's instructions, and each time `route_after_agent` finds
`tool_calls < MAX_TOOL_CALLS` and lets it through. On the request that would push the cumulative count to
`MAX_TOOL_CALLS`, `route_after_agent` finds `tool_calls >= 5` and routes to `safe_exit` instead of
`tools` — the pending tool call is never executed. `safe_exit_node` appends the friendly limit-reached
message and a `guard=loop_cap` entry to `errors`, then `safe_exit → END` ends the turn. The loop
terminated by design, not by a crash or an unhandled exception.


---
## Write-Up

**State schema choices.** `AgentState` (a `TypedDict`) has four fields: `messages` (the conversation,
via `add_messages`), and three small scalars/lists — `tool_calls`, `retries`, `errors` — chosen because
the spec calls for keeping state minimal (ids/counters, not large payloads). Nothing here stores a full
tool-result payload separately from the message log; `ToolMessage`s already carry that, and the counters
are just aggregates over it.

**Reducer rationale.** `add_messages` was chosen over a bare list because LangGraph node functions each
return only *their* delta (e.g. one new `AIMessage`), not the accumulated history — without a reducer,
each node's return value would silently overwrite the whole conversation instead of extending it.
`operator.add` on `tool_calls`/`retries` sums each node's small delta into a running total across the
thread, which is exactly what the loop-cap guard needs to compare against `MAX_TOOL_CALLS`.
`operator.add` on `errors` (a `list[str]`) is Python list concatenation, giving an append-only log of
short structured error strings, as the spec requires.

**Guard design.** The loop cap lives in a conditional-edge function (`route_after_agent`), not in a node,
because only conditional edges choose the next node in LangGraph. `tool_calls` is incremented inside
`agent_node` the moment the LLM *proposes* new tool calls, before they run; since LangGraph applies a
node's returned delta before evaluating its outgoing conditional edge, the guard always compares against
the freshly-updated total and can reject an over-limit request *before* it executes, routing to a
dedicated `safe_exit` node that appends a friendly, user-visible message plus a structured
`guard=loop_cap ...` error entry, then exits via `END`.

Reconciling "tools never raise" (this repo's `customer_tools.py` convention) with the assignment's
structured `errors` requirement: tools stay non-raising and return an `"ERROR:<CLASS>: <detail>"`
sentinel string (`CLASS` is `VALIDATION` or `TRANSIENT`) instead of a plain error message. `agent_node`
parses the trailing block of `ToolMessage`s each time it runs and turns any `ERROR:` sentinel into a
structured `errors` entry — and counts `TRANSIENT` ones toward `retries`. Because `add`/`multiply` are
pure and deterministic, every failure mode they can hit is a `VALIDATION` error (fatal, never retried,
surfaced to the user in plain language) — so `retries` stays `0` across every scenario in this notebook.
That's a deliberate scope boundary, not a bug: the retry policy exists (`VALIDATION` → fatal and
non-retryable; `TRANSIENT` → would be retried up to a small bound before escalating to fatal), but this
notebook's tools have no I/O and thus nothing transient to retry. `src/chains/memory_agent.py`'s
`try/except` around its Notion API call is this repo's real example of the kind of I/O-bound,
transient-failure-prone call the `TRANSIENT` branch is designed for — a natural candidate to wire into a
future version of this graph as an additional tool.

**Checkpointing verification.** Verified two ways: (1) Component 4 runs two separate `.invoke()` calls on
one `thread_id` and confirms the second turn's message count exceeds the first, proving history was
reloaded from the checkpoint rather than lost; (2) Acceptance Test 4 compiles the graph with
`interrupt_before=["tools"]`, invokes once to pause execution before the pending tool call runs
(`get_state(config).next == ("tools",)`), then resumes with `app.invoke(None, config=config)` on the same
`thread_id` and confirms the paused tool call's `id` matches the `tool_call_id` on the `ToolMessage`
produced after resuming — direct evidence that LangGraph replayed from the exact paused checkpoint rather
than restarting.

**A note on the spec's worked example.** The challenge document's Acceptance Test 1 states "Add 2.5 and
7, then multiply by 3" should return `27.0`. That's arithmetically inconsistent —
`(2.5 + 7) * 3 = 28.5`. This notebook and its offline test suite validate the mathematically correct
result instead of the spec's literal number.
